# Лабораторная работа № 6

В данной лабораторной работе мы изучим ансамбли моделей машинного обучения.

In [37]:
import numpy as np
import pandas as pd
from typing import Dict, Tuple
from scipy import stats
from IPython.display import Image
from io import StringIO 
from IPython.display import Image
import graphviz 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_graphviz
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score 
from sklearn.metrics import roc_curve, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline 
sns.set(style="ticks")

## Задание

* Выберите набор данных (датасет) для решения задачи классификации или регресии.
* В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.
* С использованием метода train_test_split разделите выборку на обучающую и тестовую.
* Обучите следующие ансамблевые модели:
  - одну из моделей группы стекинга.
  - модель многослойного персептрона. По желанию, вместо библиотеки scikit-learn возможно использование библиотек TensorFlow, PyTorch или других аналогичных библиотек.
  - (дополнительно) двумя методами на выбор из семейства МГУА (один из линейных методов COMBI / MULTI + один из нелинейных методов MIA / RIA) с использованием библиотеки gmdh (В настоящее время библиотека МГУА не позволяет решать задачу классификации).
* Оцените качество моделей с помощью одной из подходящих для задачи метрик. Сравните качество полученных моделей.
* В телегамм-канале потока ИУ5 в теме ТМО_МГУА напишите обратную связь по использованию библиотеки gmdh:
  - обнаруженные баги с приложением скриншотов ошибок, за каждый найденный баг +1 балл на экзамене;
  - опечатки в документации или учебном пособии МГУА;
  - возникшие вопросы или трудности при установке и использовании библиотеки;
  - любая другая информация (критика, предложения по улучшению и тд).

## Датасет

Для лабораторной работы выберем датасет : [ссылка](https://www.kaggle.com/datasets/govindaramsriram/energy-consumption-dataset-linear-regression). Перед нами задача регрессии - оценить потребление электроэнергии по ряду признаков.

На странице датасета датасет заранее разделен на обучающую и тестовые выборки. Чтобы обеспечить выполнение задания лабораторной работы, оба файла объеденены обратно - мы повторно разобъем их методом train_test_split.

In [38]:
data = pd.read_csv('data/energy_consumption/energy_data.csv', sep=",")

In [39]:
# Форма датасета

data.shape

(1100, 7)

In [40]:
# Статистика по датасету

data.describe()

,Square Footage,Number of Occupants,Appliances Used,Average Temperature,Energy Consumption
count,1100.000000,1100.000000,1100.000000,1100.000000,1100.000000
mean,25500.527273,48.268182,25.730000,22.559745,4168.191273
std,14236.955632,29.127624,14.116209,7.122357,924.278723
min,560.000000,1.000000,1.000000,10.050000,1683.950000
25%,13203.750000,22.000000,13.000000,16.365000,3510.460000
50%,25785.500000,47.000000,26.000000,22.810000,4189.690000
75%,37536.750000,73.000000,38.000000,28.760000,4859.510000
max,49997.000000,99.000000,49.000000,34.990000,6530.600000


In [41]:
# Первые 5 строк

data.head()

,Building Type,Square Footage,Number of Occupants,Appliances Used,Average Temperature,Day of Week,Energy Consumption
0,Residential,7063,76,10,29.84,Weekday,2713.95
1,Commercial,44372,66,45,16.72,Weekday,5744.99
2,Industrial,19255,37,17,14.30,Weekend,4101.24
3,Residential,13265,14,41,32.82,Weekday,3009.14
4,Commercial,13375,26,18,11.92,Weekday,3279.17


## Предобработка

Датасет имеет категориальные признаки. Их необходимо кодировать численно. Используем one-hot кодирование.

In [42]:
data_encoded = pd.get_dummies(data, drop_first=False)

data_encoded.head()

,Square Footage,Number of Occupants,Appliances Used,Average Temperature,Energy Consumption,Building Type_Commercial,Building Type_Industrial,Building Type_Residential,Day of Week_Weekday,Day of Week_Weekend
0,7063,76,10,29.84,2713.95,False,False,True,True,False
1,44372,66,45,16.72,5744.99,True,False,False,True,False
2,19255,37,17,14.30,4101.24,False,True,False,False,True
3,13265,14,41,32.82,3009.14,False,False,True,True,False
4,13375,26,18,11.92,3279.17,True,False,False,True,False


Пропусков данных нет. Выделим целевой признак в отдельный объект.

In [43]:
target = data_encoded['Energy Consumption']
data_encoded = data_encoded.drop('Energy Consumption', axis=1)

data_encoded.head()

,Square Footage,Number of Occupants,Appliances Used,Average Temperature,Building Type_Commercial,Building Type_Industrial,Building Type_Residential,Day of Week_Weekday,Day of Week_Weekend
0,7063,76,10,29.84,False,False,True,True,False
1,44372,66,45,16.72,True,False,False,True,False
2,19255,37,17,14.30,False,True,False,False,True
3,13265,14,41,32.82,False,False,True,True,False
4,13375,26,18,11.92,True,False,False,True,False


# Разбиение датасета

Разобьем данные на две выборки: обучающую и тестовую. Пропорция: ~1/5 - тестовые данные, ~4/5 - обучающие.

In [69]:
# Разделение выборки на обучающую и тестовую
energy_X_train, energy_X_test, energy_y_train, energy_y_test = train_test_split(
    data_encoded, target, test_size=0.2, random_state=1)

## Модель стекинга

Обучим модель стекинга. Прежде чем перейти непосредственно к обучению, подготовим данные, проведя масштабирование признаков.

In [70]:
# Масштабирование
scaler = StandardScaler()

# Обучаем scaler на тренировочных данных и преобразуем их
energy_X_train_scaled = scaler.fit_transform(energy_X_train)

# Преобразуем тестовые данные (используем тот же scaler, без fit!)
energy_X_test_scaled = scaler.transform(energy_X_test)

Теперь обучим модель стекинга, используя класс StackingRegressor библиотеки scikit-learn.

In [71]:
# Определяем базовые модели
# Это модели, которые будут делать первичные предсказания
base_models = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=1)),  # случайный лес
    ('gbr', GradientBoostingRegressor(n_estimators=100, random_state=1)),  # градиентный бустинг
    ('ridge', Ridge(alpha=1.0)),  # гребневая регрессия
    ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))  # метод опорных векторов
]

# Определяем финальную модель
# Она обучается на предсказаниях базовых моделей
final_model = LinearRegression()

# Создаем стек
stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=final_model,
    cv=5,  # кросс-валидация для генерации предсказаний базовых моделей
    n_jobs=-1  # используем все ядра процессора
)

# Обучаем модель на масштабированных данных
stacking_model.fit(energy_X_train_scaled, energy_y_train)

# Делаем предсказания
y_pred_stacking = stacking_model.predict(energy_X_test_scaled)

# Оцениваем качество
mae_stacking = mean_absolute_error(energy_y_test, y_pred_stacking)
# rmse_stacking = np.sqrt(mean_squared_error(energy_y_test, y_pred_stacking))

print(f"Стекинг - MAE: {mae_stacking:.4f}")
#print(f"Стекинг - RMSE: {rmse_stacking:.4f}")

Стекинг - MAE: 0.2317


## Модель многослойного персептрона

Напишем модель многослойного персептрона с помощью библиотеки PyTorch. Основываться будем на материалах [статьи](https://machinelearningmastery.com/building-multilayer-perceptron-models-in-pytorch/).

In [72]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Подготовка данных

# Конвертируем данные в тензоры
X_train_t = torch.tensor(energy_X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(energy_y_train.values, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(energy_X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(energy_y_test.values, dtype=torch.float32).view(-1, 1)

# Создаем DataLoader для пакетной подачи данных
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

Создадим модель многослойного персептрона.

In [73]:
import torch.nn as nn

# Определим количество входных признаков
input_dim = energy_X_train_scaled.shape[1]

# Создаем модель
model = nn.Sequential(
    nn.Linear(input_dim, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

print(model)  # Посмотрим на архитектуру

Sequential(
  (0): Linear(in_features=9, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=1, bias=True)
)


Определим функцию потерь и оптимизатор.

In [74]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Пройдем цикл обучения.

In [75]:
num_epochs = 140
train_losses = []

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        # Прямой проход
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        
        # Обратный проход
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    if (epoch+1) % 20 == 0:
        print(f"Эпоха {epoch+1}/{num_epochs}, Потери: {epoch_loss/len(train_loader):.4f}")

Эпоха 20/140, Потери: 69931.3502
Эпоха 40/140, Потери: 20632.5078
Эпоха 60/140, Потери: 13306.8075
Эпоха 80/140, Потери: 9233.9701
Эпоха 100/140, Потери: 6339.7406
Эпоха 120/140, Потери: 4183.0926
Эпоха 140/140, Потери: 2743.6885


Оценим полученную модель.

In [76]:
model.eval()  # Переключаем в режим оценки
with torch.no_grad():
    y_pred_t = model(X_test_t)
    y_pred_mlp = y_pred_t.numpy().flatten()

# Считаем метрики
mae_mlp = mean_absolute_error(energy_y_test, y_pred_mlp)
# rmse_mlp = np.sqrt(mean_squared_error(energy_y_test, y_pred_mlp))

print(f"MLP - MAE: {mae_mlp:.4f}")
# print(f"MLP - RMSE: {rmse_mlp:.4f}")

MLP - MAE: 48.5231


## Оценка качества моделей

Сравним качество моделей по значению метрики MAE (Mean Average Error). Выбор метрики обоснован простотой интерпретации - это среднее отклонение предсказанного значения от истинного.

In [78]:
# Собираем результаты в словарь
results = {
    'Модель': ['Стекинг (sklearn)', 'MLP (PyTorch)'],
    'MAE': [mae_stacking, mae_mlp],
}

# Создаем DataFrame для красивой таблицы
df_results = pd.DataFrame(results)
print("=== Сравнение качества моделей ===")
print(df_results.to_string(index=False))

=== Сравнение качества моделей ===
           Модель       MAE
Стекинг (sklearn)  0.231674
    MLP (PyTorch) 48.523059


## Вывод

Модель стекинга продемонстрировала аномально высокую точность, что объясняется синтетической природой датасета, где зависимости между признаками и целевой переменной близки к детерминированным. Ансамбль деревьев решений (RandomForest, GradientBoosting) практически идеально восстанавливает эти зависимости.

MLP показал ожидаемо более высокую ошибку, что характерно для нейросетей при работе с малыми объемами данных (1100 строк) и без тщательной настройки гиперпараметров. Тем не менее, относительная ошибка MLP (MAE ~1.8% от среднего значения целевой переменной) также является приемлемой.